# Solutions — Performance and Best Practices (Hard 11)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_lineitem     = spark.table("samples.tpch.lineitem")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_nation       = spark.table("samples.tpch.nation")
df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")


## Solution 1 - Avoiding Recomputation on Serverless

In [ ]:
# Why: On serverless, .cache()/.persist() throw exceptions.
# The portable pattern is: write to Delta, read back.

TABLE_NAME = "workspace.default.practice_lineitem_returned"

# 1. Filter
df_returned = df_lineitem.filter(F.col("l_returnflag") == "R")
filter_count = df_returned.count()

# 2. Write as Delta table
df_returned.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

# 3. Read it back
df_readback = spark.table(TABLE_NAME)
readback_count = df_readback.count()

# 4. Build summary
result_1 = spark.createDataFrame(
    [
        ("1_filter",    "Filter l_returnflag == R",       str(filter_count)),
        ("2_write",     "Write to Delta table",           TABLE_NAME),
        ("3_readback",  "Read back and count",            str(readback_count)),
        ("4_match",     "Counts match",                   str(filter_count == readback_count)),
    ],
    ["step", "description", "value"]
)

# 5. Cleanup
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")

## Solution 2 — repartition vs coalesce

In [ ]:
# Why: repartition triggers a full shuffle; coalesce only merges adjacent partitions.
# On serverless, .rdd.getNumPartitions() is unavailable and
# spark.conf.get("spark.sql.shuffle.partitions") may return "auto".
# Keep partition_count as a string so both cases work.

df_repartitioned = df_transactions.repartition(8)
df_coalesced = df_repartitioned.coalesce(2)

# Show plans so user can see Exchange (shuffle) vs no-shuffle
df_repartitioned.explain()
df_coalesced.explain()

result_2 = spark.createDataFrame(
    [
        ("original",              spark.conf.get("spark.sql.shuffle.partitions")),
        ("after repartition(8)",  "8"),
        ("after coalesce(2)",     "2"),
    ],
    ["operation", "partition_count"]
)

## Solution 3 — Broadcast Join for Small Tables

In [ ]:
# Why: df_nation has only 25 rows - broadcasting avoids shuffling the large side.
result_3 = (
    df_customer
    .join(F.broadcast(df_nation),
          df_customer.c_nationkey == df_nation.n_nationkey)
    .select("c_custkey", "c_name", "c_mktsegment", "n_name")
)

## Solution 4 — Reading the Execution Plan

In [ ]:
# Why: explain(mode='formatted') reveals the physical plan -
# look for FileScan (with pushed filters), Exchange (shuffles), and HashAggregate.
result_4 = (
    df_transactions
    .filter(F.col("totalPrice") > 50)
    .groupBy("franchiseID", "product")
    .agg(F.round(F.sum("totalPrice"), 2).alias("total_revenue"))
)
result_4.explain(mode="formatted")

## Solution 5 — Predicate Pushdown: Filter Early

In [ ]:
# Why: Filter BEFORE the join to reduce shuffle volume.
# Spark's Catalyst often pushes filters down, but being explicit is safer.
credit_only = df_transactions.filter(F.col("paymentMethod") == "Credit Card")

result_5 = (
    credit_only
    .join(df_franchises, on="franchiseID")
    .groupBy("franchiseID", "name", "country", "product")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
    )
)

## Solution 6 — Co-partitioning Before a Join

In [ ]:
# Repartition both DataFrames on the join key to avoid a shuffle
tx_part = df_transactions.repartition(8, "franchiseID")
fr_part = df_franchises.repartition(8, "franchiseID")

result_6 = (
    tx_part
    .join(fr_part, on="franchiseID")
    .groupBy("franchiseID", F.col("name"))
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
    )
)


## Solution 7 — Never collect() Large DataFrames

In [ ]:
# Use agg() instead of collect() for summaries
max_date = df_orders.agg(F.max("o_orderdate")).collect()[0][0]

result_7 = spark.createDataFrame(
    [("agg+collect single value", str(max_date))],
    ["approach", "max_order_date"]
)


## Solution 8 — Salting for Skewed Joins

In [ ]:
# Simulate a skewed key by adding a salt column
import random

NUM_SALT = 4

# Salt the small side: replicate rows for each salt value
salted_nation = df_nation.withColumn(
    "salt", F.explode(F.array([F.lit(i) for i in range(NUM_SALT)]))
)

# Salt the large side: assign a random salt value
salted_lineitem = (
    df_lineitem
    .withColumn("salt", (F.rand() * NUM_SALT).cast("int"))
    .join(df_orders.select("o_orderkey", "o_custkey"),
          F.col("l_orderkey") == F.col("o_orderkey"))
    .join(df_customer.select("c_custkey", "c_nationkey"),
          F.col("o_custkey") == F.col("c_custkey"))
    .withColumn("join_key", F.concat_ws("_", F.col("c_nationkey").cast("string"), F.col("salt").cast("string")))
)

salted_nation2 = salted_nation.withColumn(
    "join_key", F.concat_ws("_", F.col("n_nationkey").cast("string"), F.col("salt").cast("string"))
)

result_8 = (
    salted_lineitem
    .join(salted_nation2, on="join_key")
    .groupBy("l_shipmode", "n_name")
    .agg(
        F.round(F.sum("l_extendedprice"), 2).alias("total_revenue"),
        F.count("*").alias("line_count"),
    )
    .drop("n_name")
    .groupBy("l_shipmode")
    .agg(
        F.round(F.sum("total_revenue"), 2).alias("total_revenue"),
        F.sum("line_count").alias("line_count"),
    )
    .limit(1)
)
